# Credit Card Fraud Detection — Modelling (v2)

Rewrite of the original modelling notebook. **The EDA notebook is unchanged and still applies.**

What changed and why:

| Area | Before | Now |
|---|---|---|
| Threshold | Hardcoded `0.12` (validation search actually found ~0.89) | Always taken from the computed cost curve |
| Threshold data | 71 validation frauds | Out-of-fold predictions on train+val (~400 frauds) |
| "Tuned" XGBoost | Hand-picked, and the weakest XGBoost variant in CV | Real `RandomizedSearchCV` on PR-AUC (XGBoost and LightGBM) |
| Features | Raw scaled `Time` | `log1p(Amount)`, hour-of-day as sin/cos |
| Model comparison | 5-fold, single run, F2 at 0.5 | 5x3 repeated CV, PR-AUC + recall at precision >= 50% |
| Final model | Single XGBoost | Stacked ensemble (RF+SMOTE, XGB+SMOTE, tuned XGB, tuned LightGBM) |
| Probabilities | Distorted by SMOTE / `scale_pos_weight` | Calibrated by the stacker (logistic regression on logits) |
| Test reporting | Point estimates on 71 frauds | Bootstrap 95% CIs, plus a time-based robustness check |
| SHAP | Probabilities and SHAP rows misaligned | Same indices used for features, labels and scores |
| Export | Pickled pipeline + config | Artifact folder + `scoring.py` (`FraudScorer`) ready for the web backend |

In [6]:
# 1. Imports and configuration
import os, json, time, warnings, joblib
# Prevent Windows multiprocessing deadlocks
os.environ['LOKY_MAX_CPU_COUNT'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy.stats import loguniform, randint, uniform

from sklearn.base import clone
from sklearn.model_selection import (StratifiedKFold, RepeatedStratifiedKFold,
                                     train_test_split, RandomizedSearchCV, cross_val_predict)
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import calibration_curve
from sklearn.metrics import (precision_recall_curve, average_precision_score, roc_auc_score,
                             confusion_matrix, fbeta_score, precision_score, recall_score)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import lightgbm as lgb
import shap

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110

# ----- Configuration (one place to change things) -----
SEED = 42
FAST_MODE = True        # True = fast verified run; False = extended 5x3 repeats
N_SPLITS = 5
N_REPEATS = 1 if FAST_MODE else 3
TUNE_ITER = 4 if FAST_MODE else 30
RF_TREES = 60 if FAST_MODE else 200
N_BOOT = 200 if FAST_MODE else 1000
FP_COST = 5.0            # review cost (USD) of one false alarm
TEST_SIZE = 0.15
ARTIFACT_DIR = 'artifacts'

np.random.seed(SEED)
print(f'xgboost {xgb.__version__} | lightgbm {lgb.__version__} | shap {shap.__version__}')
print(f'FAST_MODE={FAST_MODE}  repeats={N_REPEATS}  tune_iter={TUNE_ITER}')


xgboost 3.4.1 | lightgbm 4.7.0 | shap 0.54.1
FAST_MODE=True  repeats=1  tune_iter=4


In [7]:
%%writefile scoring.py
"""Shared feature engineering + inference wrapper.

The notebook and the web backend both import from this file, so training and
serving always use identical features.
"""
import json
import joblib
import numpy as np
import pandas as pd

V_COLS = [f'V{i}' for i in range(1, 29)]
FEATURES = ['Amount_log', 'Hour_sin', 'Hour_cos'] + V_COLS


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Stateless feature engineering (safe to apply before splitting)."""
    out = df.copy()
    out['Amount_log'] = np.log1p(out['Amount'].clip(lower=0))
    hour = (out['Time'] % 86400) / 3600.0
    out['Hour_sin'] = np.sin(2 * np.pi * hour / 24.0)
    out['Hour_cos'] = np.cos(2 * np.pi * hour / 24.0)
    return out[FEATURES]


def _logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


class FraudScorer:
    """Loads the exported artifacts and scores raw transactions."""

    def __init__(self, artifact_dir: str = 'artifacts'):
        self.models = joblib.load(f'{artifact_dir}/base_models.joblib')   # dict name -> pipeline
        self.stacker = joblib.load(f'{artifact_dir}/stacker.joblib')
        with open(f'{artifact_dir}/model_config.json') as f:
            self.config = json.load(f)
        self.base_order = self.config['base_models']

    def score(self, raw_df: pd.DataFrame) -> np.ndarray:
        """Calibrated fraud probability for each row (needs Time, Amount, V1..V28)."""
        X = engineer_features(raw_df)
        Z = np.column_stack([_logit(self.models[n].predict_proba(X)[:, 1]) for n in self.base_order])
        return self.stacker.predict_proba(Z)[:, 1]

    def risk_level(self, probs: np.ndarray) -> np.ndarray:
        bands = self.config['risk_bands']
        return np.where(probs >= bands['high'], 'High',
                        np.where(probs >= bands['medium'], 'Medium', 'Low'))

    def predict(self, raw_df: pd.DataFrame) -> pd.DataFrame:
        p = self.score(raw_df)
        return pd.DataFrame({
            'fraud_probability': p,
            'flagged': p >= self.config['threshold'],
            'risk_level': self.risk_level(p),
        }, index=raw_df.index)

Overwriting scoring.py


In [8]:
from scoring import engineer_features, FEATURES, V_COLS, _logit
print('scoring.py imported. Model features:', FEATURES[:4], '...', len(FEATURES), 'total')

scoring.py imported. Model features: ['Amount_log', 'Hour_sin', 'Hour_cos', 'V1'] ... 31 total


## 2. Data loading, de-duplication and the train+val / test split

In [9]:
# 2. Load, de-duplicate, split
data_path = 'creditcard.csv'
if not os.path.exists(data_path):
    candidate_paths = [
        'data/creditcard.csv',
        '../data/creditcard.csv',
        '../creditcard.csv',
        '../../data/creditcard.csv',
        r'c:\Users\HP\OneDrive\Documents\New folder\credit-card-fraud-detection\data\creditcard.csv'
    ]
    for p in candidate_paths:
        if os.path.exists(p):
            data_path = p
            break
print(f'Using dataset: {data_path}')

df_raw = pd.read_csv(data_path)
n_dups = df_raw.duplicated().sum()
df_clean = df_raw.drop_duplicates().reset_index(drop=True)
print(f'Raw: {len(df_raw):,} rows | duplicates dropped: {n_dups:,} | clean: {len(df_clean):,}')
print(f'Frauds: {df_clean.Class.sum():,} ({df_clean.Class.mean()*100:.3f}%)')

# One split: "dev" (train + validation, used for CV, tuning, stacking, threshold) and a held-out test set
# touched exactly once at the end.  Using CV on dev instead of a fixed 15% validation slice gives us
# ~400 frauds to select the threshold with, instead of 71.
dev_df, test_df = train_test_split(df_clean, test_size=TEST_SIZE, stratify=df_clean['Class'], random_state=SEED)
dev_df, test_df = dev_df.reset_index(drop=True), test_df.reset_index(drop=True)

X_dev, y_dev = engineer_features(dev_df), dev_df['Class'].values
X_test, y_test = engineer_features(test_df), test_df['Class'].values
amt_dev, amt_test = dev_df['Amount'].values, test_df['Amount'].values

print('=' * 60)
print(f'Dev  (train+val): {len(X_dev):,} txns | {y_dev.sum()} frauds')
print(f'Held-out test:    {len(X_test):,} txns | {y_test.sum()} frauds')
print('=' * 60)
ratio = (len(y_dev) - y_dev.sum()) / y_dev.sum()
print(f'Imbalance ratio {ratio:.0f}:1  | sqrt ratio {np.sqrt(ratio):.1f}')

Using dataset: ../data/creditcard.csv
Raw: 284,807 rows | duplicates dropped: 1,081 | clean: 283,726
Frauds: 473 (0.167%)
Dev  (train+val): 241,167 txns | 402 frauds
Held-out test:    42,559 txns | 71 frauds
Imbalance ratio 599:1  | sqrt ratio 24.5


## 3. Preprocessing and metric helpers

In [17]:
# 3. Preprocessing + helpers
def make_pre():
    # Only Amount_log needs scaling (matters for Logistic Regression); trees ignore it.
    return ColumnTransformer(
        [('scale', StandardScaler(), ['Amount_log']),
         ('keep', 'passthrough', [c for c in FEATURES if c != 'Amount_log'])],
        remainder='drop')

PRE_FEATURE_NAMES = ['Amount_log'] + [c for c in FEATURES if c != 'Amount_log']


def recall_at_precision(y_true, probs, min_precision=0.5):
    prec, rec, _ = precision_recall_curve(y_true, probs)
    ok = prec >= min_precision
    return rec[ok].max() if ok.any() else 0.0


def cost_curve(y_true, probs, amounts, thresholds, fp_cost=FP_COST):
    """Total cost = value of missed frauds + review cost of false alarms."""
    costs = []
    for t in thresholds:
        flagged = probs >= t
        fn_cost = amounts[(y_true == 1) & (~flagged)].sum()
        fp_total = ((y_true == 0) & flagged).sum() * fp_cost
        costs.append(fn_cost + fp_total)
    return np.array(costs)


def bootstrap_ci(y_true, probs, threshold, n_boot=N_BOOT, seed=SEED):
    """Stratified bootstrap 95% CI for PR-AUC, recall and precision."""
    rng = np.random.default_rng(seed)
    pos, neg = np.where(y_true == 1)[0], np.where(y_true == 0)[0]
    out = {'PR-AUC': [], 'Recall': [], 'Precision': []}
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        yt, pr = y_true[idx], probs[idx]
        out['PR-AUC'].append(average_precision_score(yt, pr))
        out['Recall'].append(recall_score(yt, pr >= threshold, zero_division=0))
        out['Precision'].append(precision_score(yt, pr >= threshold, zero_division=0))
    return {k: (np.percentile(v, 2.5), np.percentile(v, 97.5)) for k, v in out.items()}

print('Helpers ready.')

Helpers ready.


## 4. Hyperparameter tuning (XGBoost and LightGBM)

Tuned on **PR-AUC** (`average_precision`) with 3-fold stratified CV on the dev set. Early-stopping-style behaviour is
approximated by searching `n_estimators` and `learning_rate` together, and `scale_pos_weight` is searched over
softer values than the full imbalance ratio (the full ratio usually over-weights frauds).

Note: scores from the tuned models inside the CV table below are slightly optimistic because the same dev data was used
to pick the parameters. The held-out test set at the end is the unbiased check.

In [18]:
# 4a. Tune XGBoost
spw_options = sorted({1, 5, int(round(np.sqrt(ratio))), 100})
tune_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

xgb_pipe = ImbPipeline([('pre', make_pre()),
                        ('clf', xgb.XGBClassifier(tree_method='hist', eval_metric='aucpr',
                                                  random_state=SEED, n_jobs=-1))])
xgb_space = {
    'clf__n_estimators': randint(200, 800),
    'clf__learning_rate': loguniform(0.02, 0.15),
    'clf__max_depth': randint(3, 8),
    'clf__min_child_weight': randint(1, 10),
    'clf__subsample': uniform(0.6, 0.4),
    'clf__colsample_bytree': uniform(0.5, 0.5),
    'clf__gamma': uniform(0, 2),
    'clf__reg_lambda': loguniform(0.5, 20),
    'clf__scale_pos_weight': spw_options,
}
t0 = time.time()
search_xgb = RandomizedSearchCV(xgb_pipe, xgb_space, n_iter=TUNE_ITER, scoring='average_precision',
                                cv=tune_cv, random_state=SEED, n_jobs=1, refit=False)
search_xgb.fit(X_dev, y_dev)
print(f'XGBoost search done in {time.time()-t0:.0f}s | best CV PR-AUC = {search_xgb.best_score_:.4f}')
print({k.replace('clf__', ''): (round(float(v), 4) if isinstance(v, (float, np.floating)) else int(v)) for k, v in search_xgb.best_params_.items()})

XGBoost search done in 407s | best CV PR-AUC = 0.8605
{'colsample_bytree': 0.6669, 'gamma': 0.2857, 'learning_rate': 0.0742, 'max_depth': 7, 'min_child_weight': 2, 'n_estimators': 543, 'reg_lambda': 10.7794, 'scale_pos_weight': 5, 'subsample': 0.6003}


In [14]:
# 4b. Tune LightGBM
lgb_pipe = ImbPipeline([('pre', make_pre()),
                        ('clf', lgb.LGBMClassifier(subsample_freq=1, random_state=SEED,
                                                   n_jobs=-1, verbose=-1))])
lgb_space = {
    'clf__n_estimators': randint(200, 800),
    'clf__learning_rate': loguniform(0.02, 0.15),
    'clf__num_leaves': randint(8, 64),
    'clf__min_child_samples': randint(5, 60),
    'clf__subsample': uniform(0.6, 0.4),
    'clf__colsample_bytree': uniform(0.5, 0.5),
    'clf__reg_lambda': loguniform(0.5, 20),
    'clf__scale_pos_weight': spw_options,
}
t0 = time.time()
search_lgb = RandomizedSearchCV(lgb_pipe, lgb_space, n_iter=TUNE_ITER, scoring='average_precision',
                                cv=tune_cv, random_state=SEED, n_jobs=1, refit=False)
search_lgb.fit(X_dev, y_dev)
print(f'LightGBM search done in {time.time()-t0:.0f}s | best CV PR-AUC = {search_lgb.best_score_:.4f}')
print({k.replace('clf__', ''): (round(float(v), 4) if isinstance(v, (float, np.floating)) else int(v)) for k, v in search_lgb.best_params_.items()})

LightGBM search done in 11234s | best CV PR-AUC = 0.8568
{'colsample_bytree': 0.6062, 'learning_rate': 0.0288, 'min_child_samples': 25, 'n_estimators': 360, 'num_leaves': 19, 'reg_lambda': 4.7739, 'scale_pos_weight': 1, 'subsample': 0.7728}


## 5. Candidate models

In [16]:
# 5. Build candidate pipelines
def tuned(base_pipe, search):
    p = clone(base_pipe)
    p.set_params(**search.best_params_)
    return p

candidates = {
    'Logistic Regression (balanced)': ImbPipeline([
        ('pre', make_pre()),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=SEED))]),
    'RF + SMOTE': ImbPipeline([
        ('pre', make_pre()),
        ('smote', SMOTE(sampling_strategy=0.1, random_state=SEED)),
        ('clf', RandomForestClassifier(n_estimators=RF_TREES, max_depth=12, min_samples_leaf=2,
                                       random_state=SEED, n_jobs=-1))]),
    'XGB + SMOTE': ImbPipeline([
        ('pre', make_pre()),
        ('smote', SMOTE(sampling_strategy=0.1, random_state=SEED)),
        ('clf', xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8,
                                  colsample_bytree=0.8, tree_method='hist', eval_metric='aucpr',
                                  random_state=SEED, n_jobs=-1))]),
    'XGB (tuned)': tuned(xgb_pipe, search_xgb),
    'LightGBM (tuned)': tuned(lgb_pipe, search_lgb),
}
BASE_MODELS = ['RF + SMOTE', 'XGB + SMOTE', 'XGB (tuned)', 'LightGBM (tuned)']
print('Candidates:', list(candidates))
print('Ensemble members:', BASE_MODELS)

Candidates: ['Logistic Regression (balanced)', 'RF + SMOTE', 'XGB + SMOTE', 'XGB (tuned)', 'LightGBM (tuned)']
Ensemble members: ['RF + SMOTE', 'XGB + SMOTE', 'XGB (tuned)', 'LightGBM (tuned)']


## 6. Repeated stratified CV benchmark (5 x 3)

Reported per model: PR-AUC, ROC-AUC and **recall at precision >= 50%** (a threshold-free way to compare models at a
comparable operating point, unlike F2 at a fixed 0.5 cutoff). Out-of-fold predictions from the first repeat are kept
for stacking, calibration and threshold selection.

In [ ]:
# 6. Repeated CV benchmark
rcv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
splits = list(rcv.split(X_dev, y_dev))

fold_pr, fold_roc, fold_rec50 = {}, {}, {}
oof = {name: np.zeros(len(y_dev)) for name in candidates}

print('=' * 78)
for name, pipe in candidates.items():
    t0 = time.time()
    prs, rocs, recs = [], [], []
    for i, (tr, va) in enumerate(splits):
        m = clone(pipe).fit(X_dev.iloc[tr], y_dev[tr])
        p = m.predict_proba(X_dev.iloc[va])[:, 1]
        prs.append(average_precision_score(y_dev[va], p))
        rocs.append(roc_auc_score(y_dev[va], p))
        recs.append(recall_at_precision(y_dev[va], p, 0.5))
        if i < N_SPLITS:          # first repeat -> out-of-fold predictions
            oof[name][va] = p
    fold_pr[name], fold_roc[name], fold_rec50[name] = np.array(prs), np.array(rocs), np.array(recs)
    print(f'{name:<32} PR-AUC {np.mean(prs):.4f} +/- {np.std(prs):.4f} | '
          f'Recall@P50 {np.mean(recs):.3f} | {time.time()-t0:.0f}s')
print('=' * 78)

In [ ]:
# 6b. Summary table + paired comparison against the best model
rows = []
best_name = max(fold_pr, key=lambda n: fold_pr[n].mean())
for n in candidates:
    diff = fold_pr[n] - fold_pr[best_name]
    rows.append({'Model': n,
                 'PR-AUC': f'{fold_pr[n].mean():.4f} +/- {fold_pr[n].std():.4f}',
                 'ROC-AUC': f'{fold_roc[n].mean():.4f} +/- {fold_roc[n].std():.4f}',
                 'Recall@P>=50%': f'{fold_rec50[n].mean():.3f} +/- {fold_rec50[n].std():.3f}',
                 'PR-AUC gap vs best (paired)': 'best' if n == best_name else f'{diff.mean():+.4f} +/- {diff.std():.4f}',
                 'mean': fold_pr[n].mean()})
df_bench = pd.DataFrame(rows).sort_values('mean', ascending=False).drop(columns='mean')
print(df_bench.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
order = list(df_bench['Model'])
for ax, d, title in [(axes[0], fold_pr, 'PR-AUC'), (axes[1], fold_rec50, 'Recall at precision >= 50%')]:
    means = [d[n].mean() for n in order]; stds = [d[n].std() for n in order]
    ax.barh(order, means, xerr=stds, color='#2b5c8f', edgecolor='black', alpha=0.85)
    ax.invert_yaxis(); ax.set_title(title, fontweight='bold'); ax.set_xlim(0, 1)
plt.suptitle(f'{N_SPLITS}x{N_REPEATS} repeated stratified CV (error bars = std across folds)', fontweight='bold')
plt.tight_layout(); plt.show()
print('Differences smaller than the fold std are within noise - do not over-read small gaps.')

## 7. Stacking and calibration

The ensemble members' out-of-fold probabilities are converted to logits and combined by a plain (unweighted-class)
logistic regression. This does two jobs at once: it **blends** the models and it **calibrates** the output, so the
score shown in the web tool behaves like a real probability (SMOTE and `scale_pos_weight` otherwise distort it).
The stacker itself is evaluated with a further cross-validation over the out-of-fold matrix.

In [ ]:
# 7. Stack + calibrate
Z_dev = np.column_stack([_logit(oof[n]) for n in BASE_MODELS])
stacker = LogisticRegression(C=1.0, max_iter=1000)

stack_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 1)
stack_oof = cross_val_predict(stacker, Z_dev, y_dev, cv=stack_cv, method='predict_proba')[:, 1]
mean_blend_oof = np.mean([oof[n] for n in BASE_MODELS], axis=0)

summary = {n: average_precision_score(y_dev, oof[n]) for n in candidates}
summary['Simple mean blend'] = average_precision_score(y_dev, mean_blend_oof)
summary['Stacked + calibrated'] = average_precision_score(y_dev, stack_oof)
print('Pooled out-of-fold PR-AUC (dev set, first CV repeat):')
for k, v in sorted(summary.items(), key=lambda kv: -kv[1]):
    print(f'  {k:<32} {v:.4f}')

fig, ax = plt.subplots(figsize=(5.5, 5))
frac_pos, mean_pred = calibration_curve(y_dev, stack_oof, n_bins=10, strategy='quantile')
ax.plot(mean_pred, frac_pos, 'o-', color='#d9383a', label='Stacked + calibrated')
frac_b, mean_b = calibration_curve(y_dev, np.clip(oof['XGB (tuned)'], 0, 1), n_bins=10, strategy='quantile')
ax.plot(mean_b, frac_b, 's--', color='gray', label='XGB (tuned) raw')
ax.plot([0, 1], [0, 1], 'k:', label='Perfect'); ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Predicted probability'); ax.set_ylabel('Observed fraud rate'); ax.legend()
ax.set_title('Calibration (out-of-fold, log scale)', fontweight='bold')
plt.tight_layout(); plt.show()

## 8. Cost-based decision threshold (from out-of-fold predictions)

Cost = value of missed frauds + review cost of false alarms. The curve is smoothed with a small moving average so the
chosen threshold is not the result of one lucky spike. **The threshold used everywhere below is the computed one.**

In [ ]:
# 8. Threshold selection
thresholds = np.linspace(0.005, 0.995, 400)
costs = cost_curve(y_dev, stack_oof, amt_dev, thresholds)
smooth = np.convolve(costs, np.ones(7) / 7, mode='same')
smooth[:3], smooth[-3:] = costs[:3], costs[-3:]
t_star = float(thresholds[np.argmin(smooth)])
cost_star = float(cost_curve(y_dev, stack_oof, amt_dev, [t_star])[0])
cost_default = float(cost_curve(y_dev, stack_oof, amt_dev, [0.5])[0])
cost_no_model = float(amt_dev[y_dev == 1].sum())

# Medium-risk (watch-list) band: highest threshold that still gives >= 95% out-of-fold recall
rec_curve = np.array([recall_score(y_dev, stack_oof >= t) for t in thresholds])
ok = np.where(rec_curve >= 0.95)[0]
t_medium = float(thresholds[ok.max()]) if len(ok) else t_star / 2
t_medium = min(t_medium, t_star)

pred = stack_oof >= t_star
print('=' * 66)
print('THRESHOLD SELECTION (out-of-fold, dev set)')
print('=' * 66)
print(f'Selected threshold t*:          {t_star:.4f}')
print(f'Watch-list (medium) threshold:  {t_medium:.4f}')
print(f'Cost with no model:             ${cost_no_model:,.2f}')
print(f'Cost at t = 0.50:               ${cost_default:,.2f}')
print(f'Cost at t*:                     ${cost_star:,.2f}')
print(f'Recall at t*: {recall_score(y_dev, pred):.3f} | Precision at t*: {precision_score(y_dev, pred):.3f} | '
      f'False alarms: {int(((y_dev==0)&pred).sum()):,}')
print('=' * 66)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.8))
ax1.plot(thresholds, costs, color='#d9383a', alpha=0.35, label='Cost')
ax1.plot(thresholds, smooth, color='#d9383a', lw=2.3, label='Cost (smoothed)')
ax1.axvline(t_star, color='black', ls='--', label=f't* = {t_star:.3f}')
ax1.axvline(0.5, color='gray', ls=':', label='t = 0.5')
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f'${int(x):,}'))
ax1.set_xlabel('Threshold'); ax1.set_ylabel('Total cost'); ax1.legend()
ax1.set_title('A. Operational cost vs threshold', fontweight='bold')
prec_c = np.array([precision_score(y_dev, stack_oof >= t, zero_division=0) for t in thresholds])
ax2.plot(thresholds, rec_curve, color='#d9383a', label='Recall'); ax2.plot(thresholds, prec_c, color='#2b5c8f', label='Precision')
ax2.axvline(t_star, color='black', ls='--'); ax2.axvline(t_medium, color='orange', ls='--', label='Watch-list threshold')
ax2.set_xlabel('Threshold'); ax2.legend(); ax2.set_title('B. Recall / precision trade-off', fontweight='bold')
plt.tight_layout(); plt.show()

## 9. Final fit and one-shot evaluation on the held-out test set

Base models are refit on all of dev, the stacker is fit on the out-of-fold logits, and the test set is scored once.

In [ ]:
# 9. Fit everything on dev, evaluate on test
t0 = time.time()
final_models = {n: clone(candidates[n]).fit(X_dev, y_dev) for n in BASE_MODELS}
stacker.fit(Z_dev, y_dev)
print(f'Final models fitted in {time.time()-t0:.0f}s')

test_base = {n: final_models[n].predict_proba(X_test)[:, 1] for n in BASE_MODELS}
Z_test = np.column_stack([_logit(test_base[n]) for n in BASE_MODELS])
test_probs = stacker.predict_proba(Z_test)[:, 1]

print('Held-out test PR-AUC per component:')
for n in BASE_MODELS:
    print(f'  {n:<22} {average_precision_score(y_test, test_base[n]):.4f}')
test_pr = average_precision_score(y_test, test_probs)
print(f'  {"Stacked + calibrated":<22} {test_pr:.4f}')

In [ ]:
# 9b. Test report at t* vs default, with bootstrap CIs
def report(th):
    pr = test_probs >= th
    tn, fp, fn, tp = confusion_matrix(y_test, pr).ravel()
    return dict(tp=tp, fn=fn, fp=fp, tn=tn,
                recall=tp / (tp + fn), precision=tp / max(tp + fp, 1),
                f2=fbeta_score(y_test, pr, beta=2),
                cost=float(cost_curve(y_test, test_probs, amt_test, [th])[0]))

r_def, r_star = report(0.5), report(t_star)
ci = bootstrap_ci(y_test, test_probs, t_star)
test_cost_none = float(amt_test[y_test == 1].sum())

table = pd.DataFrame({
    'Metric': ['Frauds caught', 'Missed frauds', 'False alarms', 'Precision', 'Recall', 'F2', 'Total cost ($)'],
    'Threshold 0.50': [f"{r_def['tp']}/{r_def['tp']+r_def['fn']}", r_def['fn'], r_def['fp'],
                       f"{r_def['precision']:.3f}", f"{r_def['recall']:.3f}", f"{r_def['f2']:.3f}", f"{r_def['cost']:,.2f}"],
    f'Threshold t*={t_star:.3f}': [f"{r_star['tp']}/{r_star['tp']+r_star['fn']}", r_star['fn'], r_star['fp'],
                       f"{r_star['precision']:.3f}", f"{r_star['recall']:.3f}", f"{r_star['f2']:.3f}", f"{r_star['cost']:,.2f}"],
})
print('=' * 70); print('HELD-OUT TEST SET'); print('=' * 70)
print(table.to_string(index=False))
print(f'No-model cost on this test set: ${test_cost_none:,.2f}')
print('-' * 70)
print(f"PR-AUC {test_pr:.4f}  (95% CI {ci['PR-AUC'][0]:.3f} - {ci['PR-AUC'][1]:.3f})")
print(f"Recall at t*    95% CI {ci['Recall'][0]:.3f} - {ci['Recall'][1]:.3f}")
print(f"Precision at t* 95% CI {ci['Precision'][0]:.3f} - {ci['Precision'][1]:.3f}")
print(f'(test set has only {int(y_test.sum())} frauds, so wide intervals are expected)')

In [ ]:
# 9c. Confusion matrices + PR curve
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, th, name, cmap in [(axes[0], 0.5, 'Default t = 0.50', 'Blues'), (axes[1], t_star, f't* = {t_star:.3f}', 'Reds')]:
    cm = confusion_matrix(y_test, test_probs >= th)
    sns.heatmap(cm, annot=True, fmt=',d', cmap=cmap, cbar=False, ax=ax,
                xticklabels=['Genuine', 'Fraud'], yticklabels=['Genuine', 'Fraud'])
    ax.set_title(name, fontweight='bold'); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
pc, rc, _ = precision_recall_curve(y_test, test_probs)
axes[2].plot(rc, pc, color='#d9383a', lw=2.3, label=f'PR-AUC {test_pr:.3f}')
axes[2].scatter([r_star['recall']], [r_star['precision']], color='black', s=90, zorder=5, label='Operating point t*')
axes[2].axhline(y_test.mean(), color='gray', ls='--', label='Random baseline')
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision'); axes[2].legend(); axes[2].set_title('Test PR curve', fontweight='bold')
plt.tight_layout(); plt.show()

## 10. Time-based robustness check

The random split above lets the model see transactions from the same time window it is tested on. Here we train on
the earliest ~75% of the 2-day window and test on the latest part, which is closer to how a deployed model is used.
This is a sanity check only (fewer frauds, so noisy) and is not used for any selection.

In [ ]:
# 10. Temporal hold-out
df_t = df_clean.sort_values('Time').reset_index(drop=True)
cut = df_t['Time'].quantile(0.75)
tr_t, te_t = df_t[df_t['Time'] <= cut], df_t[df_t['Time'] > cut]
print(f'Train: {len(tr_t):,} txns / {tr_t.Class.sum()} frauds | Test (later): {len(te_t):,} txns / {te_t.Class.sum()} frauds')

if te_t.Class.sum() >= 10:
    rows = []
    for n in ['XGB (tuned)', 'LightGBM (tuned)', 'RF + SMOTE']:
        m = clone(candidates[n]).fit(engineer_features(tr_t), tr_t.Class.values)
        p = m.predict_proba(engineer_features(te_t))[:, 1]
        rows.append({'Model': n, 'Temporal PR-AUC': round(average_precision_score(te_t.Class, p), 4),
                     'Recall@P>=50%': round(recall_at_precision(te_t.Class.values, p, 0.5), 3)})
    print(pd.DataFrame(rows).to_string(index=False))
    print('A large drop versus the random-split numbers suggests sensitivity to time/drift.')
else:
    print('Too few frauds in the later window for a meaningful temporal check.')

## 11. Explainability (SHAP)

SHAP explains the tuned **XGBoost** component (the ensemble itself is a blend of several models). The explained sample
contains every test-set fraud plus a random set of genuine transactions, and **the same row indices** are used for the
SHAP values, labels and ensemble scores, so each waterfall plot matches the transaction it claims to show.
SHAP values are in XGBoost log-odds units.

In [ ]:
# 11. SHAP with aligned indices
xgb_final = final_models['XGB (tuned)']
pre_fit = xgb_final.named_steps['pre']
X_test_t = pd.DataFrame(pre_fit.transform(X_test), columns=PRE_FEATURE_NAMES)

rng = np.random.default_rng(SEED)
fraud_idx = np.where(y_test == 1)[0]
legit_idx = rng.choice(np.where(y_test == 0)[0], size=min(1500, int((y_test == 0).sum())), replace=False)
sample_idx = np.sort(np.concatenate([fraud_idx, legit_idx]))          # ONE index array used everywhere

explainer = shap.TreeExplainer(xgb_final.named_steps['clf'])
sv = explainer(X_test_t.iloc[sample_idx])
y_s, p_s = y_test[sample_idx], test_probs[sample_idx]                  # aligned with sv rows

plt.figure(figsize=(9, 5.5))
shap.summary_plot(sv, X_test_t.iloc[sample_idx], show=False, max_display=12)
plt.title('Global feature drivers (all test frauds + random genuine sample)', fontweight='bold')
plt.tight_layout(); plt.show()

tp_rows = np.where((y_s == 1) & (p_s >= t_star))[0]
fp_rows = np.where((y_s == 0) & (p_s >= t_star))[0]
if len(tp_rows):
    r = tp_rows[np.argmax(p_s[tp_rows])]
    exp_tp = sv[r, :, 1] if len(sv.shape) == 3 else sv[r]
    shap.plots.waterfall(exp_tp, max_display=9, show=False)
    plt.title(f'True positive | ensemble risk = {p_s[r]:.3f}', fontweight='bold'); plt.tight_layout(); plt.show()
if len(fp_rows):
    r = fp_rows[np.argmax(p_s[fp_rows])]
    exp_fp = sv[r, :, 1] if len(sv.shape) == 3 else sv[r]
    shap.plots.waterfall(exp_fp, max_display=9, show=False)
    plt.title(f'False positive | ensemble risk = {p_s[r]:.3f}', fontweight='bold'); plt.tight_layout(); plt.show()


## 12. Export artifacts for the web backend

In [ ]:
# 12. Export
for adir in [ARTIFACT_DIR, os.path.join('..', ARTIFACT_DIR)]:
    os.makedirs(adir, exist_ok=True)
    joblib.dump({n: final_models[n] for n in BASE_MODELS}, f'{adir}/base_models.joblib')
    joblib.dump(stacker, f'{adir}/stacker.joblib')

config = {
    'model_type': 'Stacked ensemble (logistic-regression stacker on logits)',
    'base_models': BASE_MODELS,
    'features': FEATURES,
    'required_input_columns': ['Time', 'Amount'] + V_COLS,
    'threshold': t_star,
    'risk_bands': {'high': t_star, 'medium': t_medium},
    'fp_review_cost_usd': FP_COST,
    'threshold_source': 'cost-optimal on out-of-fold predictions (dev set)',
    'metrics_test': {
        'pr_auc': float(test_pr), 'pr_auc_ci95': [float(x) for x in ci['PR-AUC']],
        'recall_at_threshold': float(r_star['recall']), 'recall_ci95': [float(x) for x in ci['Recall']],
        'precision_at_threshold': float(r_star['precision']), 'precision_ci95': [float(x) for x in ci['Precision']],
        'n_test_frauds': int(y_test.sum()),
    },
    'library_versions': {'xgboost': xgb.__version__, 'lightgbm': lgb.__version__},
}
for adir in [ARTIFACT_DIR, os.path.join('..', ARTIFACT_DIR)]:
    with open(f'{adir}/model_config.json', 'w') as f:
        json.dump(config, f, indent=2)
print('Saved to artifacts dirs:', os.listdir(ARTIFACT_DIR))


In [ ]:
# 12b. Verify the exported artifacts reproduce the notebook's scores (this is exactly what the web backend will do)
from scoring import FraudScorer
scorer = FraudScorer(ARTIFACT_DIR)
check = test_df.drop(columns='Class').head(500)
served = scorer.score(check)
assert np.allclose(served, test_probs[:500], atol=1e-6), 'Exported pipeline does not match notebook scores!'
print('Round-trip check passed: FraudScorer reproduces notebook probabilities.')
print(scorer.predict(check).head(8).to_string())
print(f"\nRisk levels on 500 test rows: {pd.Series(scorer.predict(check)['risk_level']).value_counts().to_dict()}")